# Canopy Change Along the Swannanoa River Valley: Before vs. After Hurricane Helene

**Area of interest:** Swannanoa River corridor from Black Mountain, NC to its confluence with the French Broad River in Asheville (Biltmore Village) — one of the most severely flooded and wind-damaged stretches in Buncombe County during Hurricane Helene (landfall 2024-09-27).

**Method:** NDVI (Normalized Difference Vegetation Index) computed from NASA HLS (Harmonized Landsat Sentinel-2) 30 m surface reflectance, comparing a clear-sky scene shortly before Helene to one shortly after, to map vegetation/canopy loss from wind damage, flooding, and landslides.

**Data source:** NASA LP DAAC, HLSL30/HLSS30 v2.0, accessed via `earthaccess` (NASA Earthdata Login required).

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path
from src.ndvi import compute_ndvi, ndvi_difference, align_to
from src.plotting import plot_before_after_diff, plot_change_histogram
from src.search_download import load_aoi_bbox

AOI_PATH = "../config/aoi_swannanoa.geojson"
aoi_bounds = load_aoi_bbox(AOI_PATH)
aoi_bounds

## 1. Acquire imagery

Run once from the project root (needs a free NASA Earthdata Login — https://urs.earthdata.nasa.gov/):

```bash
# Pre-Helene: last clear scene before landfall (2024-09-27)
python src/search_download.py --start 2024-08-15 --end 2024-09-26 --product HLSL30 --max-cloud 15

# Post-Helene: first clear scene after landfall
python src/search_download.py --start 2024-09-28 --end 2024-10-20 --product HLSL30 --max-cloud 15
```

If HLSL30 (Landsat) has no low-cloud scene in the post-storm window, fall back to `--product HLSS30` (Sentinel-2), which revisits more frequently. Downloaded `.tif` bands land in `data/raw/`.

Fill in the actual downloaded filenames below once you've run the search.

In [ ]:
RAW = Path("../data/raw")

# TODO: replace with the actual HLS granule filenames after running search_download.py
PRE_RED = RAW / "HLS.L30.T17SNV.2024xxx.v2.0.B04.tif"
PRE_NIR = RAW / "HLS.L30.T17SNV.2024xxx.v2.0.B05.tif"
PRE_FMASK = RAW / "HLS.L30.T17SNV.2024xxx.v2.0.Fmask.tif"

POST_RED = RAW / "HLS.L30.T17SNV.2024yyy.v2.0.B04.tif"
POST_NIR = RAW / "HLS.L30.T17SNV.2024yyy.v2.0.B05.tif"
POST_FMASK = RAW / "HLS.L30.T17SNV.2024yyy.v2.0.Fmask.tif"

## 2. Compute NDVI (pre and post), with cloud/shadow masking

In [ ]:
pre_ndvi = compute_ndvi(str(PRE_RED), str(PRE_NIR), str(PRE_FMASK), aoi_bounds=aoi_bounds)
post_ndvi = compute_ndvi(str(POST_RED), str(POST_NIR), str(POST_FMASK), aoi_bounds=aoi_bounds)

print("Pre-Helene mean NDVI: ", float(pre_ndvi.mean()))
print("Post-Helene mean NDVI:", float(post_ndvi.mean()))

## 3. Difference map and canopy loss quantification

In [ ]:
diff = ndvi_difference(pre_ndvi, post_ndvi)
post_aligned = align_to(pre_ndvi, post_ndvi)

fig = plot_before_after_diff(pre_ndvi, post_aligned, diff)
fig.savefig("../outputs/figures/ndvi_before_after_diff.png", dpi=200, bbox_inches="tight")

In [ ]:
fig, loss_pct = plot_change_histogram(diff, loss_threshold=-0.15)
fig.savefig("../outputs/figures/ndvi_change_histogram.png", dpi=200, bbox_inches="tight")
print(f"{loss_pct:.1f}% of AOI pixels show ΔNDVI < -0.15 (notable canopy/vegetation loss)")

## 4. (Stretch) Overlay flood extent or land cover change

Candidate supplementary datasets, if time allows:
- USGS/NOAA post-Helene flood extent polygons (high-water marks, flood inundation mapping)
- NC Emergency Management landslide inventory for Buncombe County
- NLCD land cover change product for broader context

These would be reprojected to match the NDVI grid and overlaid on the `ΔNDVI` raster to distinguish flood-driven vegetation loss from wind-driven canopy damage upslope.